In [ ]:
import rootutils

_ = rootutils.setup_root(rootutils.find_root(), pythonpath=True, cwd=True)

In [ ]:
import math
from pathlib import Path
from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from tqdm.auto import tqdm
import torch

from gjepa.utils.config import load_and_resolve_config
from gjepa.datasets.node_level import load_graph
from gjepa.utils.pyg import create_transform

sns.set_theme("notebook")

import warnings

warnings.filterwarnings("ignore", "is_categorical_dtype")
warnings.filterwarnings("ignore", "use_inf_as_na")
warnings.filterwarnings("ignore", "figure layout has changed to tight")
warnings.filterwarnings("ignore", "unique with argument that is not not")

In [ ]:
ds_config_flies = list(Path("../config/dataset/").iterdir())
ds_configs = [load_and_resolve_config(cfg_file) for cfg_file in ds_config_flies]

In [ ]:
torch.manual_seed(2137)

In [ ]:
label_counts = []
for cfg in tqdm(ds_configs):
    if cfg["name"] == "ogbn-arxiv":
        continue
    data = load_graph(
        root_dir=Path(cfg["root_dir"]),
        name=cfg["name"],
        transform=create_transform(cfg["transforms"]),
        pre_transform=create_transform(cfg["pre_transforms"]),
    )
    label_counts.append({"dataset": cfg["name"], "counts": Counter(data.y.tolist())})

In [ ]:
num_ds = len(label_counts)

num_cols = 2
num_rows = math.ceil(num_ds / num_cols)
fig, axes = plt.subplots(num_rows, num_cols, figsize=(6 * num_cols, num_rows * 4), squeeze=False)

for l_count, ax in zip(label_counts, axes.flatten()):
    x, y = list(l_count["counts"].keys()), list(l_count["counts"].values())
    sns.barplot(x=x, y=y, ax=ax)
    ax.set(title=l_count["dataset"])

# fig.suptitle("Label distribution")
fig.tight_layout()
plt.show()